In [9]:
import requests
from bs4 import BeautifulSoup
import csv
import re

In [10]:
def extract_state_stats(url):
    """
    Extract newspaper location (state) and count statistics from a Library of Congress page.
    
    Args:
        url: The URL of the page to scrape
    
    Returns:
        A list of dictionaries containing 'state' and 'count'
    """
    # Fetch the page
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    
    # Parse the HTML
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Find the list container
    list_container = soup.select_one('div.index-listbox ul')
    
    if not list_container:
        print("Could not find the list container")
        return []
    
    # Extract all list items
    results = []
    list_items = list_container.find_all('li')
    
    for item in list_items:
        # Find the anchor tag
        link = item.find('a')
        if not link:
            continue
        
        # Extract state (from span.label or the text before the count)
        state_span = link.find('span', class_='label')
        if state_span:
            state = state_span.get_text(strip=True)
        else:
            # Fallback: get all text and remove the count part
            full_text = link.get_text(strip=True)
            # The count is typically in brackets at the end
            state = re.sub(r'\s*\[\d+\]\s*$', '', full_text)
        
        # Extract count (from span.count)
        count_span = link.find('span', class_='count')
        if count_span:
            count = count_span.get_text(strip=True)
            # Remove brackets if present
            count = count.strip('[]')
        else:
            # Fallback: extract from the text using regex
            count_match = re.search(r'\[(\d+)\]', link.get_text())
            count = count_match.group(1) if count_match else '0'
        
        results.append({
            'state': state,
            'count': count
        })
    
    return results

In [17]:
def save_to_csv(data, filename='state_statistics.csv'):
    """
    Save the extracted data to a CSV file.
    
    Args:
        data: List of dictionaries with 'state' and 'count' keys
        filename: Name of the output CSV file
    """
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['state', 'count'])
        writer.writeheader()
        writer.writerows(data)
    print(f"Data saved to {filename}")

def main():
    # The URL to scrape
    url = 'https://www.loc.gov/collections/chronicling-america/index/location_state/?dl=page&ops=AND&qs=%22chinese+student%22&searchType=advanced&sp=1'
    
    print("Fetching data from Library of Congress...")
    data = extract_state_stats(url)
    
    if data:
        print(f"\nExtracted {len(data)} state entries:\n")
        
        # Print the first few entries as a preview
        print(f"{'State':<100} {'Count':>10}")
        print("-" * 90)
        for i, entry in enumerate(data[:5]):
            print(f"{entry['state']:<100} {entry['count']:>10}")
        if len(data) > 5:
            print(f"... and {len(data) - 5} more entries")
        
        # Save to CSV
        save_to_csv(data)
        
        # Print summary statistics
        total_count = sum(int(entry['count']) for entry in data)
        print(f"\nTotal states: {len(data)}")
        print(f"Total article count: {total_count}")
    else:
        print("No data extracted. Please check the URL or page structure.")

In [18]:
if __name__ == '__main__':
    main()

Fetching data from Library of Congress...

Extracted 53 state entries:

State                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

ValueError: invalid literal for int() with base 10: '1,452'